# Library 剩餘24段｜先 dry-run，等待確認
Rosie 13 / Princess 11。已接受六句只原樣複製，不生成。Bakery及Park不在輸入包。免費Colab GPU可能有時數及資源限制，無付費API。

In [ ]:
import sys, subprocess, os
os.environ['HF_HUB_DISABLE_IMPLICIT_TOKEN']='1'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
subprocess.check_call(['apt-get','-qq','update'])
subprocess.check_call(['apt-get','-qq','install','-y','espeak-ng','sox','libsndfile1'])
subprocess.check_call([sys.executable,'-m','pip','install','-q','qwen-tts==0.1.1','transformers==4.57.3','accelerate==1.12.0','kokoro==0.9.4','soundfile','ipywidgets>=8,<9','huggingface-hub>=0.34,<1'])
print('套件安裝完成。若Colab提示套件已載入，請重新啟動執行階段，再從上傳格繼續。')

## 2 上傳 library-remaining24-inputs.zip

In [ ]:
from google.colab import files
from pathlib import Path
import sys,uuid,zipfile,io
up=files.upload()
raw=next(iter(up.values()))
WORK=Path('/content/library24_'+uuid.uuid4().hex[:8])
with zipfile.ZipFile(io.BytesIO(raw)) as z:
 for name in z.namelist():
  p=(WORK/name).resolve()
  if not p.is_relative_to(WORK.resolve()):raise ValueError('Unsafe ZIP')
 z.extractall(WORK)
sys.path.insert(0,str(WORK))
import remaining_runner as runner
OUTPUT=Path('/content/library30_'+uuid.uuid4().hex[:8])

## 3 Dry-run｜執行後先停，把清單回傳確認

In [ ]:
from IPython.display import display,Markdown
plan=runner.generate(WORK,OUTPUT,run=False)
display(Markdown('|角色|ID|英文原文|\n|---|---|---|\n'+'\n'.join('|'+a['role']+'|'+a['lineId']+'|'+a['text']+'|' for a in plan['jobs'])))
print('24 jobs / 0 generated. Protected six hashes verified.')

## 4 等家長確認24段清單後才啟用

In [ ]:
RUN_GENERATION=False
CONFIRMATION='' # 確認後填 CONFIRM_LIBRARY_REMAINING_24
if RUN_GENERATION:
 result=runner.generate(WORK,OUTPUT,run=True,confirmation=CONFIRMATION)
else:print('未生成，等待清單確認。')

## 5 生成後逐句聽音，再下載獨立30句副本

In [ ]:
from IPython.display import Audio,display
import shutil,json
if not (OUTPUT/'generated-manifest.json').exists():print('尚未生成；本格不推論。')
else:
 result=runner.read(OUTPUT/'generated-manifest.json')
 for a in result['assets'].values():
  print(a['role'],a['voiceId'],a['text'],'[已接受六句]' if a['assetId'] in runner.read(WORK/'manifest.json')['protectedAssets'] else '[待人工驗收]')
  display(Audio(filename=str(OUTPUT/a['src'])))
 if len(result['assets'])!=30:raise ValueError('尚未完成30句，請保留輸出並續跑')
 files.download(shutil.make_archive('/content/library30-review','zip',OUTPUT))

## 驗收與授權
已接受六句維持原WAV與SHA256。新24句仍需人工檢查童真、溫柔、漏詞及聲線一致性，未自動核准。Qwen/Kokoro Apache-2.0；參考音僅依家長核准用途，權利來源未另行獨立稽核，勿公開參考包。mouth events為台詞層級及音量動畫，不是精準音素嘴型。